# Module 1 · OpenAI SDK basics through a BFSI use case
**Use case:** Loan Application Status Assistant  ·  ⏱ ~15 min

In this notebook we will:
1. Initialise the client (raw OpenAI SDK vs **aisuite**)
2. Send **system** and **user** instructions
3. Pass a customer request and display the response
4. Generate a simple **structured output** (JSON validated with Pydantic)

> 🗣️ *Speaker cue:* everything today runs on the OpenAI API. aisuite is a thin wrapper that gives one `provider:model` interface, so the code stays tiny and readable.

In [1]:
import os, sys, json
sys.path.append("..")                      # so we can import utils.py and harness/

from dotenv import load_dotenv
import aisuite as ai
from utils import print_html, print_json, show_messages, show_tool_call, show_trace, show_usage, compare

load_dotenv("../.env", override=True)
MODEL = os.getenv("MODEL", "openai:gpt-4o-mini")   # aisuite format -> "provider:model"
client = ai.Client()                               # reads OPENAI_API_KEY from the environment

print_html({"model": MODEL, "client": "aisuite.Client() ready"}, title="Setup", color="gray")

model,openai:gpt-4o-mini
client,aisuite.Client() ready


## Step 1 · Initialise the client: raw OpenAI SDK first
This is the official OpenAI Python SDK. Note the model name is just `gpt-4o-mini`.

In [2]:
from openai import OpenAI

oai = OpenAI()
raw = oai.chat.completions.create(
    model=MODEL.split(":", 1)[1],
    messages=[{"role": "user", "content": "In one sentence, what is a loan application?"}],
)
print_html(raw.choices[0].message.content, title="Raw OpenAI SDK")

### The same call with aisuite
Same request and response shape (`choices[0].message.content`). The only difference is the model string: `"openai:gpt-4o-mini"`.
Switching to another provider later means changing **one string**.

In [3]:
response = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": "In one sentence, what is a loan application?"}],
)
print_html(response.choices[0].message.content, title="aisuite → OpenAI", color="green")

## Step 2 · System + user instructions
The **system** message sets the assistant's role, scope and rules. The **user** message is the customer's request.

In [4]:
SYSTEM_PROMPT = """You are the Loan Service Assistant for Demo Bank.
- Help customers with questions about their loan applications.
- Be concise (max 4 sentences), polite and factual.
- Never ask for PAN, Aadhaar, card or account numbers.
- You cannot approve or reject loans."""

customer_request = "Hi, I applied for a home loan last month. What usually happens after I submit the application?"

messages = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user", "content": customer_request},
]
response = client.chat.completions.create(model=MODEL, messages=messages, temperature=0.2)

show_messages(messages + [response.choices[0].message], title="Loan Service Assistant")
show_usage(response)

prompt_tokens,86
completion_tokens,80
total_tokens,166


## Step 3 · The limitation: the model doesn't know *your* data
Ask about a specific application. The model has never seen Demo Bank's loan system, so it can only say "I don't know", or worse, **make something up**.

In [5]:
messages = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user", "content": "What is the status of my application LA-1002?"},
]
response = client.chat.completions.create(model=MODEL, messages=messages, temperature=0.2)
print_html(response.choices[0].message.content, title="Without access to bank systems", color="red")

> 🗣️ *Speaker cue:* this is the bridge to Module 2. We need to give the model a safe, **approved** way to look things up: a **tool**.

## Step 4 · Structured output
Applications need data, not prose. We ask for JSON (`response_format={"type": "json_object"}` is passed straight through aisuite to OpenAI) and **validate** it with Pydantic.

In [6]:
from typing import Literal, Optional
from pydantic import BaseModel, Field

class CustomerRequest(BaseModel):
    intent: Literal["loan_status", "missing_documents", "decision_request",
                    "complaint", "general_question", "other"]
    application_id: Optional[str] = Field(None, description="Loan ID like LA-1234, if mentioned")
    sentiment: Literal["positive", "neutral", "negative"]
    needs_human: bool = Field(description="True for complaints or requests to change a decision")
    summary: str = Field(description="One-line summary for a loan officer")

CLASSIFY_PROMPT = f"""You classify customer messages for a bank's loan desk.
Return ONLY a JSON object that matches this JSON schema:
{json.dumps(CustomerRequest.model_json_schema(), indent=1)}"""

def classify(text: str) -> CustomerRequest:
    resp = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "system", "content": CLASSIFY_PROMPT},
                  {"role": "user", "content": text}],
        response_format={"type": "json_object"},
        temperature=0,
    )
    return CustomerRequest.model_validate_json(resp.choices[0].message.content)

result = classify("Where is my loan LA-1001? It's been 3 weeks!")
print_html(result, title="Structured output (validated)", color="purple")

intent,loan_status
application_id,LA-1001
sentiment,negative
needs_human,False
summary,Customer inquires about the status of loan LA-1001 after 3 weeks.


In [7]:
samples = [
    "What is the status of LA-1003?",
    "Which documents are pending for my personal loan LA-1002?",
    "Please approve my loan LA-1001 today, I need it urgently.",
    "This is unacceptable, LA-1004 was rejected without any reason!",
    "What is an EMI?",
]
rows = [{"message": s, **classify(s).model_dump()} for s in samples]
print_html(rows, title="Classifying a batch of customer messages", color="purple")

message,intent,application_id,sentiment,needs_human,summary
What is the status of LA-1003?,loan_status,LA-1003,neutral,False,Customer inquires about the status of loan application LA-1003.
Which documents are pending for my personal loan LA-1002?,missing_documents,LA-1002,neutral,False,Customer inquires about pending documents for personal loan LA-1002.
"Please approve my loan LA-1001 today, I need it urgently.",decision_request,LA-1001,positive,True,Customer urgently requests loan approval for LA-1001.
"This is unacceptable, LA-1004 was rejected without any reason!",complaint,LA-1004,negative,True,Customer is unhappy about the rejection of loan application LA-1004 without explanation.
What is an EMI?,general_question,None,neutral,False,Customer is asking for clarification on what EMI means.


> 💡 **Going further:** OpenAI also supports `response_format={"type": "json_schema", "json_schema": {..., "strict": True}}`, which guarantees the output matches the schema exactly.

## ✅ Takeaways
- One client and one call. The **system prompt** defines role and rules.
- `choices[0].message.content` is the answer. `usage` tells you the cost.
- Structured output turns free text into data your application can **route** on.
- The model alone **cannot know** a customer's loan status → **Module 2: tools**.